# Suspended IDs

This notebook grabs additional data to flesh out suspect user IDs. PRAW is not available for suspended accounts, meaning we need to source data elsewhere. This process is different than the active account info collection because you can use Reddit.

In [ ]:
# ARCHIVED draft of the Arctic Shift post-history pull (replaced by p2_data_cleaning/collection_scripts/tier4_arctic_full_history.py).
# Libraries.
#import libraries
import pandas as pd
import numpy as np
import os 
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
import re
from string import punctuation
import functools
import operator
from dotenv import load_dotenv
import praw
import prawcore
from prawcore.exceptions import NotFound
import pprint
import ast
import pprint
import os 
import itertools
import pickle
import requests
import time 
from requests_ratelimiter import LimiterSession
import pprint
import json
import base64
from itertools import islice
import pyarrow as pa
import pyarrow.parquet as pq
import random

In [ ]:
# Accounts from the first collection (User_Profiles.parquet).
users = pd.read_parquet('User_Profiles.parquet', engine='pyarrow')

In [ ]:
user_list=list(users['username'])

In [ ]:
# 10 random accounts as a test set.
u = random.sample(user_list, 10) 

In [ ]:
# Version 1: page through each account's posts. In this version the page's posts are never added to all_posts,
# so the result is empty; version 2 below fixes that.
def get_historic_data(name_list):
    session = LimiterSession(per_second=5)
    user_submissions = {}
    seen_ids = set()  # track seen post ids across all users
    
    for name in name_list:
        print(name)
        all_posts = []
        after = None
        
        while True:
            arctic_url = f'https://arctic-shift.photon-reddit.com/api/posts/search?author={name}&limit=100'
            if after:
                arctic_url += f'&after={after}'
            
            retries = 0
            success = False
            
            while retries < 10:
                print(f'Retries are at: {retries}')
                try:
                    response = session.get(arctic_url)
                    
                    if response.status_code == 200:
                        api_return = response.__dict__
                        data_content = json.loads(api_return
                                                  .get('_content')
                                                  .decode('utf-8'))
                        
                        posts = data_content['data']
                        
                        
                        if len(posts) < 100:
                            success = True
                            break
                        
                        after = posts[-1]['created_utc']
                        success = True
                        break
                    
                    else:
                        retries += 1
                        print(f'API response: {response.status_code}')
                
                except Exception as e:
                    print(f"Hit unexpected error with class {e.__class__}:{e}")
                    retries += 1
            
            if not success or len(posts) < 100:
                break
        
        user_submissions[name] = all_posts
    
    # build DataFrame from all users
    rows = []
    for outer_key, records in user_submissions.items():
        for record in records:
            meta = record.get('_meta', {})
            row = {'outer_key': outer_key} | meta | {k: v for k, v in record.items() if k != '_meta'}
            rows.append(row)

    df = pd.DataFrame(rows)
    
    # deduplicate after building DataFrame as a safety net
    
    #df.to_pickle('postsample.pkl')
    
    return df

In [ ]:
test=get_historic_data(u)

In [ ]:
test.shape

In [ ]:
# Write a DataFrame to data/<dir_name>/ as parquet partitioned by account; nested values become JSON strings.
def save_to_parquet(df,dir_name):
    path=f'data/{dir_name}'
    os.makedirs(path, exist_ok=True)
    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
            df[col] = df[col].apply(
                lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
            )
        if df[col].apply(lambda x: isinstance(x, (int, float, bool))).all():
            try:
                df[col] = df[col].astype(str)
            except:
                pass
                
    table = pa.Table.from_pandas(df)
    pq.write_to_dataset(
        table,
        root_path=path,
        partition_cols=['outer_key'],
        compression='zstd'
    )

In [ ]:
# Version 2: same paging, and each account's posts are written straight to data/<dir_name>/ (partitioned by account).
# Like the other drafts it stops at the first short page and uses after = last created_utc, which can skip items;
# the final pipeline fixed both.
def get_historic_data(name_list,dir_name):
    session = LimiterSession(per_second=5)
    user_submissions = {}
    
    for name in name_list:
        print(name)
        all_posts = []
        after = None
        
        while True:
            arctic_url = f'https://arctic-shift.photon-reddit.com/api/posts/search?author={name}&limit=100'
            if after:
                arctic_url += f'&after={after}'
            
            retries = 0
            success = False
            
            while retries < 10:
                print(f'Retries are at: {retries}')
                try:
                    response = session.get(arctic_url)
                    
                    if response.status_code == 200:
                        api_return = response.__dict__
                        data_content = json.loads(api_return
                                                  .get('_content')
                                                  .decode('utf-8'))
                        
                        posts = data_content['data']
                        all_posts.extend(posts)
                        
                        if len(posts) < 100:
                            success = True
                            break
                        
                        after = posts[-1]['created_utc']
                        success = True
                        break
                    
                    else:
                        retries += 1
                        print(f'API response: {response.status_code}')
                
                except Exception as e:
                    print(f"Hit unexpected error with class {e.__class__}:{e}")
                    raise e
            
            if not success or len(posts) < 100:
                break


        
        rows = []
    
        for record in all_posts:
            meta = record.get('_meta', {})
            row =  meta | {k: v for k, v in record.items() if k != '_meta'}
            rows.append(row)
        
        df = pd.DataFrame(rows)
        df['outer_key']=name
        for col in df.columns:
            if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
                df[col] = df[col].apply(
                    lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
                )
            types = df[col].dropna().apply(type).unique()
            if len(types) > 1:
                df[col] = df[col].apply(
                    lambda x: str(x) if not isinstance(x, (bool, type(None))) else x
                )
        
        pq.write_to_dataset(
            pa.Table.from_pandas(df),
            root_path=f'data/{dir_name}',
            partition_cols=['outer_key'],
            compression='zstd'
        )
            
     

    
    
    

In [ ]:
# Test run on the 10 accounts; this version returns nothing, it writes files.
test=get_historic_data(u,'posts')

In [ ]:
test.head(2)